In [ ]:
# import sys
# sys.path.append("..")

In [1]:
import pandas as pd

df_raw = pd.read_csv("../data/raw/questions.csv")
print(f"Shape: {df_raw.shape} \nColumns: {list(df_raw.columns)}")

df_raw.head()

Shape: (404351, 6) 
Columns: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate']


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


# Drop and Missing Values
Since our data is not sequential (e.g. time series), we can drop the `id` column since it brings no analysis power.

Moreover, as we can see from the cell below the number of null values is negletable compared to the number of rows (only $3$), hence they are dropped as well.

The small dataset containing the Null values is memorized in case is necessary. After dropping the rows the index is reseted, so the index doesn't contain gaps.

Finally, columns `qid1` and `qid2` are also dropped since the same exact question can be referenced to a different `qid`, this may result in a problem for the clustering algorithm.

In [2]:
q1 = df_raw[["qid1", "question1"]]
q2 = df_raw[["qid2", "question2"]]

q1 = q1.rename(columns={"qid1": "qid",
                        "question1": "question"})

q2 = q2.rename(columns={"qid2": "qid",
                        "question2": "question"})

q1_with_q2 = pd.concat([q1, q2]).dropna()

most_duplicated = q1_with_q2.groupby("question")["qid"].nunique().sort_values(ascending=False)
print(most_duplicated[most_duplicated > 1].head())

question
What are the best ways to lose weight?                                             161
How can you look at someone's private Instagram account without following them?    120
How can I lose weight quickly?                                                     111
What's the easiest way to make money online?                                        88
Can you see who views your Instagram?                                               79
Name: qid, dtype: int64


In [3]:
df = df_raw.drop(columns=["id", "qid1", "qid2"])

print("Number of Null Values Before")
print(df.isna().sum())


print(f"\nShape before: {df.shape}")
df_null = df[df.isna().any(axis=1)]

df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Number of Null Values After")
print(f"Shape after: {df.shape}\n")
print(df.isna().sum())
df_null.head()

Number of Null Values Before
question1       1
question2       2
is_duplicate    0
dtype: int64

Shape before: (404351, 3)
Number of Null Values After
Shape after: (404348, 3)

question1       0
question2       0
is_duplicate    0
dtype: int64


,question1,question2,is_duplicate
105796,How can I develop android app?,NaN,0
201871,How can I create an Android app?,NaN,0
363416,NaN,My Chinese name is Haichao Yu. What English na...,0


In [4]:
df["question1_norm"] = df["question1"]
df["question2_norm"] = df["question2"]

for col in ["question1_norm", "question2_norm"]:
    df[col] = df[col].str.normalize("NFKC")
    df[col] = df[col].str.lower()
    df[col] = df[col].str.split().str.join(" ")

target = df.pop("is_duplicate")
df["is_duplicate"] = target
df.head()

,question1,question2,question1_norm,question2_norm,is_duplicate
0,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,what is the story of kohinoor (koh-i-noor) dia...,what would happen if the indian government sto...,0
2,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,why am i mentally very lonely? how can i solve...,find the remainder when [math]23^{24}[/math] i...,0
4,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,"which one dissolve in water quikly sugar, salt...",which fish would survive in salt water?,0


In [ ]:
# 1 Question ids: one integer per distinct text_norm, and pairs rewritten with those ids.
# 2 Pair checks: remove pairs where both sides are the same question, remove repeated pairs (smaller id first), and handle pairs that have both labels
# 3 Ground-truth groups: connected components of the pairs labelled duplicate, with the group size distribution.
# 4 Tuning and test split: about 20% / 80% of the questions, split by connected component of the whole pair graph and with a fixed seed, so that no question appears on both sides.
# 5 Save: questions.parquet (id, text_raw, text_norm, group, split), pairs.parquet (ids and is_duplicate), checked by reading them back.
# 6 Numbers for the report: class balance (36.9% duplicates), how many pairs each question appears in, and every count printed above.

In [5]:
q1_norm = df["question1_norm"].rename("question")
q2_norm = df["question2_norm"].rename("question")

df_unique_questions = pd.concat( [q1_norm, q2_norm], ignore_index=True ).drop_duplicates().reset_index(drop=True)
print(df_unique_questions.head(10))
print(len(df_unique_questions))

0    what is the step by step guide to invest in sh...
1    what is the story of kohinoor (koh-i-noor) dia...
2    how can i increase the speed of my internet co...
3    why am i mentally very lonely? how can i solve...
4    which one dissolve in water quikly sugar, salt...
5    astrology: i am a capricorn sun cap moon and c...
6                                  should i buy tiago?
7                       how can i be a good geologist?
8                      when do you use シ instead of し?
9    motorola (company): can i hack my charter moto...
Name: question, dtype: object
537174


In [6]:
unique_ids = df_unique_questions.to_dict()
unique_ids = dict((v, k) for k, v in unique_ids.items())

df_pair_questions = pd.DataFrame({ "qid1": df["question1_norm"].map(unique_ids), 
                                   "qid2": df["question2_norm"].map(unique_ids), 
                                   "is_duplicate": df["is_duplicate"] })
print(df_pair_questions.head(10))

   qid1    qid2  is_duplicate
0     0  290379             0
1     1   16735             0
2     2  290380             0
3     3  290381             0
4     4  290382             0
5     5  290383             1
6     6  290384             0
7     7  290385             1
8     8  273498             0
9     9  290386             0


# Trivial or Incorrect Pairs
All pairs where both sides are the same identical question are trivially equal.

All repeated pairs are, e.g. `(5, 9) == (9, 5)`, are considered equal by default and bring no additional information

All pairs that appear with both labels is data inconsistency, therefore removed.

In [7]:
df_questions = df_pair_questions.copy()
identical_pairs = df_questions["qid1"] == df_questions["qid2"]

print(f"Number of Questions with Both Sides Equal: {identical_pairs.sum()}")

Number of Questions with Both Sides Equal: 38


In [8]:
sorted_pairs = df_questions[ ["qid1", "qid2"] ].apply( lambda row: tuple(sorted(row)), axis=1 )
sorted_pairs = sorted_pairs.rename("sorted_pairs")

repeated_pairs = sorted_pairs.duplicated()
print(f"Number of Equal Pair Questions: {repeated_pairs.sum()}")

Number of Equal Pair Questions: 528


In [9]:
sorted_pairs = sorted_pairs.to_frame()
sorted_pairs["is_duplicate"] = df_questions["is_duplicate"]

inconsistent_pairs = sorted_pairs.groupby("sorted_pairs")["is_duplicate"].transform("nunique") > 1

print(f"Number of Rows in pairs with inconsistent target: {inconsistent_pairs.sum()}")

Number of Rows in pairs with inconsistent target: 20


In [10]:
keep_row = ~identical_pairs & ~inconsistent_pairs & ~repeated_pairs
print(f"Number of Rows removed: {(~keep_row).sum()} and Rows kept: {keep_row.sum()}")

Number of Rows removed: 576 and Rows kept: 403772


In [11]:
remove_rows = identical_pairs | repeated_pairs | inconsistent_pairs
df_clean = df_questions[~remove_rows].copy()

df_clean[ ["qid1", "qid2"] ] = sorted_pairs.loc[~remove_rows, "sorted_pairs"].tolist()
df_clean = df_clean.reset_index(drop=True)

print(df_clean.head(10))

   qid1    qid2  is_duplicate
0     0  290379             0
1     1   16735             0
2     2  290380             0
3     3  290381             0
4     4  290382             0
5     5  290383             1
6     6  290384             0
7     7  290385             1
8     8  273498             0
9     9  290386             0


## 25 questions that only appeared in removed pairs were kept in the corpus without labels

In [ ]:
used_questions = pd.concat([df_clean["qid1"], df_clean["qid2"]]).unique()
unsed_questions = ~df_unique_questions.index.isin(used_questions)
print(unsed_questions.sum())

## Baseline Check and Groups Creation
Although accuracy won't be as good of a performance metric (F1 will be better), it's still good to establish some sort of a baseline. In this case, `63.07%` will be our baseline for accuracy.

## Ground-truth duplicate groups

Duplicate is transitive: if A~B and B~C, then A~C even when that pair is not labelled. We build groups of questions that are all duplicates of each other: one graph node per question, one edge per pair labelled duplicate, and the connected components (SciPy) are the groups. Each question gets a `group_id`, and a question without a duplicate is alone in its group.

- 448,100 groups: 60,379 with 2 or more questions, and 387,721 single questions (72%), including 25 that only appeared in removed pairs. Most groups are small, and the largest has 108 questions.
- The 148,773 labelled duplicate pairs imply 229,223 pairs (54% more). The implied pairs are used as additional ground truth, and are generated from `group_id` when needed.
- 107 pairs are labelled "not duplicate" but fall inside one group. They are kept and flagged in `conflicting_pair`, and excluded when the implied pairs are used.

In [12]:
print(df_clean.is_duplicate.value_counts())
baseline_acc = len(df_clean[df_clean.is_duplicate == 0]) / len(df_clean)
print(f"Baseline Accuracy: {baseline_acc:.2f}")
print(f"Duplicate Pairs: {1 - baseline_acc:.2f}")

is_duplicate
0    254999
1    148773
Name: count, dtype: int64
Baseline Accuracy: 0.63
Duplicate Pairs: 0.37


In [13]:
import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

all_duplicates = df_clean[df_clean["is_duplicate"] == 1]
n = len(df_unique_questions)
graph = coo_matrix( (np.ones(len(all_duplicates)), (all_duplicates["qid1"], all_duplicates["qid2"]) ), shape=(n, n))

In [ ]:
number_of_groups, group_id = connected_components(graph, directed=False)
sizes = pd.Series(group_id).value_counts()

print(f"Number of Groups: {number_of_groups} \nGroups with Size >1: {(sizes > 1).sum()}")
print("Group sizes (size: number of groups)")
print(sizes.value_counts().sort_index().head())
print(f"Largest group: {sizes.max()} questions")
print(f"Labelled duplicate pairs: {len(all_duplicates)}")
print(f"Duplicate pairs implied by the groups: {(sizes * (sizes - 1) // 2).sum()}")

Number of Groups: 448100 
Groups with Size >1: 60379
Size of the groups: count
1    387721
2     48186
3      7242
4      2239
5       998
Name: count, dtype: int64
Largest group: 108 questions
Labelled duplicate pairs: 148773
Duplicate pairs implied by the groups: 229223


In [15]:
questions = df_unique_questions.to_frame(name="text_norm")
questions.index.name = "qid"
questions = questions.reset_index()

questions["group_id"] = group_id

df_clean["conflicting_pair"] = (
    (df_clean["is_duplicate"] == 0)
    & (group_id[df_clean["qid1"]] == group_id[df_clean["qid2"]])
)
print(f"Number of Conflicting Pairs: {df_clean['conflicting_pair'].sum()}")

Number of Conflicting Pairs: 107


In [18]:
gid = sizes[sizes == 3].index[0]
questions[questions["group_id"] == gid]

,qid,text_norm,group_id
37431,37431,which are the best christmas movies of all time?,33478
66885,66885,which is the best christmas movie?,33478
415323,415323,what are the best christmas movie characters?,33478


In [19]:
all_pairs_graph = coo_matrix((np.ones(len(df_clean)), (df_clean["qid1"], df_clean["qid2"])), shape=(n, n))
n_comp, comp_id = connected_components(all_pairs_graph, directed=False)
comp_sizes = pd.Series(comp_id).value_counts()
print(n_comp, comp_sizes.max(), comp_sizes.max() / n)    # number of components, largest one, its share of all questions

200335 9934 0.018493076731189523
